# 🌿 Plant Disease Detection & Classification: CNN vs MobileNetV2 vs ResNet50 vs Vision Transformer (ViT)

A Deep Learning notebook comparing **four distinct computer vision paradigms** for multi-class plant disease classification across **38 plant/pathogen classes** using the **PlantVillage** dataset:

1. **Custom CNN (from scratch)**: 4 Convolution blocks with BatchNorm, MaxPooling, and Dropout.
2. **Vision Transformer (ViT from scratch)**: Pure multi-head self-attention operating on 16x16 image patches with learnable positional embeddings, Pre-LayerNorm, and GELU MLP blocks.
3. **MobileNetV2 (Transfer Learning)**: Inverted residual bottleneck architecture pretrained on ImageNet; lightweight and fast for edge devices.
4. **ResNet50 (Transfer Learning)**: Deep residual network pretrained on ImageNet with skip connections.

---

### 📌 Kaggle Dataset Link:
* **PlantVillage Dataset on Kaggle:** [https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset)
* *Alternative Mirror:* [https://www.kaggle.com/datasets/emmarex/plantdisease](https://www.kaggle.com/datasets/emmarex/plantdisease)

---

### 🚀 Running on Kaggle (Free GPU Instructions):
1. Open [Kaggle](https://www.kaggle.com/) and click **Create** ➔ **New Notebook**.
2. Go to **File** ➔ **Import Notebook** and upload this `.ipynb` file.
3. In the right-hand panel under **Settings**:
   * Set **Accelerator = GPU (T4 x2 or P100)**.
   * Set **Internet = On** (required to download ImageNet pretrained weights for MobileNetV2 and ResNet50).
4. Click **+ Add Input** ➔ search for **`PlantVillage Dataset`** (by *Abdallah Ali*) ➔ click **+** to add it.
5. Click **Run All**.
6. When complete, download **`results.zip`** from the **Output** panel (`/kaggle/working`) and unzip into your project directory to run the Streamlit app!


## ⚙️ 1. Environment Setup & GPU Accelerator Verification


In [ ]:
import os
import sys
import glob
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
import keras
from keras import layers

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Detected:      {len(gpus)} -> {gpus}")
if gpus:
    print(f"CUDA Accelerator Active: {tf.test.gpu_device_name()}")
else:
    print("WARNING: No GPU detected! Running on CPU will be slower.")


## 📂 2. Dataset Discovery & Path Resolution
The notebook automatically locates the color images directory whether running on Kaggle or a local environment.


In [ ]:
# Search for the PlantVillage 'color' directory in Kaggle inputs or local directories
candidates = [
    d for d in glob.glob("/kaggle/input/**/color", recursive=True) if os.path.isdir(d)
]
if not candidates:
    candidates = [
        d for d in glob.glob("/kaggle/input/**/PlantVillage", recursive=True) if os.path.isdir(d)
    ]
if not candidates:
    candidates = [
        d for d in glob.glob("data/**/color", recursive=True) if os.path.isdir(d)
    ]
if not candidates:
    candidates = [
        d for d in glob.glob("data/**/PlantVillage", recursive=True) if os.path.isdir(d)
    ]

if candidates:
    DATA_DIR = Path(candidates[0])
    print(f"Dataset found at: {DATA_DIR}")
    classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')])
    print(f"Total Disease Classes Found: {len(classes)}")
    print(f"Sample Classes: {classes[:5]}")
else:
    print("Dataset directory not found automatically. Using demo synthetic setup or custom path.")
    DATA_DIR = Path("data/PlantVillage")


## 🎛️ 3. Project Configuration & Hyperparameters


In [ ]:
# Image parameters
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 42

# Sample limit for fast training (Set to None to use full ~54,000 dataset)
MAX_PER_CLASS = 1000   # e.g., 1000 images per class for balanced, rapid convergence

# Training Epochs
CNN_EPOCHS = 20
VIT_EPOCHS = 20
HEAD_EPOCHS = 5        # Transfer learning phase 1 (frozen backbone)
FINETUNE_EPOCHS = 8    # Transfer learning phase 2 (fine-tune top layers)
FINE_TUNE_LAYERS = 30  # Number of top backbone layers to unfreeze in phase 2

# Learning Rates
LR_CNN = 1e-3
LR_VIT = 1e-3
LR_HEAD = 1e-3
LR_FINETUNE = 1e-5

# Vision Transformer Architecture Hyperparameters
VIT_PATCH_SIZE = 16          # 16x16 patch size -> (224/16)^2 = 196 patches
VIT_PROJECTION_DIM = 128     # Embedding vector dimension
VIT_NUM_HEADS = 4            # Multi-head attention heads
VIT_TRANSFORMER_LAYERS = 4   # Transformer encoder blocks
VIT_MLP_DIM = 256            # Dense hidden units in Transformer MLP
VIT_DROPOUT = 0.1

MODELS_DIR = Path("models")
OUTPUTS_DIR = Path("outputs")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")


## 🔄 4. Stratified Data Split & Balanced tf.data Pipeline
We split the data into **70% Training, 15% Validation, and 15% Test** using stratified sampling so every class is proportionally represented in all splits.


In [ ]:
def discover_images(data_dir, max_per_class=None, seed=SEED):
    rng = np.random.default_rng(seed)
    class_folders = sorted([d for d in data_dir.iterdir() if d.is_dir() and not d.name.startswith('.')])
    class_names = [d.name for d in class_folders]
    
    paths, labels = [], []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    
    for label_idx, folder in enumerate(class_folders):
        imgs = [p for p in folder.iterdir() if p.suffix.lower() in valid_exts]
        if max_per_class and len(imgs) > max_per_class:
            imgs = list(rng.choice(imgs, size=max_per_class, replace=False))
        for img_p in imgs:
            paths.append(str(img_p))
            labels.append(label_idx)
            
    return paths, labels, class_names

paths, labels, class_names = discover_images(DATA_DIR, MAX_PER_CLASS)
print(f"Total Images: {len(paths)} across {len(class_names)} classes.")

# Stratified Split (70% train, 15% val, 15% test)
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    paths, labels, test_size=0.30, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}")

# Save class names & split meta
(OUTPUTS_DIR / "class_names.json").write_text(json.dumps(class_names, indent=2))

# Compute class weights for imbalanced datasets
from sklearn.utils.class_weight import compute_class_weight
classes_arr = np.unique(y_train)
cw_values = compute_class_weight(class_weight="balanced", classes=classes_arr, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes_arr, cw_values)}
print("Class weights computed.")


### Building High-Performance `tf.data` Input Pipeline


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)  # Raw 0-255 range (models have internal rescaling)
    img.set_shape((*IMG_SIZE, 3))
    return img, label

def create_dataset(paths, labels, batch_size=BATCH_SIZE, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, np.asarray(labels, dtype=np.int32)))
    if is_training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset(X_train, y_train, is_training=True)
val_ds   = create_dataset(X_val, y_val, is_training=False)
test_ds  = create_dataset(X_test, y_test, is_training=False)
print("Data pipeline ready.")


## 🧬 5. Neural Network Architecture Implementations

### Augmentation & Custom Preprocessing Layers
All models incorporate in-model data augmentation (active only during training) and custom registered layers for seamless `.keras` saving and reloading.


In [ ]:
def build_augmentation():
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.15, fill_mode="reflect"),
            layers.RandomZoom(0.15, fill_mode="reflect"),
            layers.RandomTranslation(0.10, 0.10, fill_mode="reflect"),
            layers.RandomContrast(0.15),
        ],
        name="augmentation",
    )

# ----------------- 1. Custom CNN -----------------
def build_custom_cnn(num_classes, img_shape=IMG_SHAPE):
    inputs = keras.Input(shape=img_shape, name="image")
    x = build_augmentation()(inputs)
    x = layers.Rescaling(1.0 / 255.0, name="rescaling")(x)
    
    for filters in (32, 64, 128, 256):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
        
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="cnn")

# ----------------- 2. Vision Transformer (ViT) -----------------
@keras.saving.register_keras_serializable(package="plantdisease")
class AddPositionEmbs(layers.Layer):
    """Learnable 1D positional embedding added to patch sequences."""
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = int(num_patches)
        self.projection_dim = int(projection_dim)

    def build(self, input_shape):
        self.pos_embedding = self.add_weight(
            name="pos_embedding",
            shape=(1, self.num_patches, self.projection_dim),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        return x + self.pos_embedding

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return cfg

def build_vit(num_classes, img_shape=IMG_SHAPE, patch_size=VIT_PATCH_SIZE,
              projection_dim=VIT_PROJECTION_DIM, num_heads=VIT_NUM_HEADS,
              transformer_layers=VIT_TRANSFORMER_LAYERS, mlp_dim=VIT_MLP_DIM, dropout=VIT_DROPOUT):
    num_patches = (img_shape[0] // patch_size) * (img_shape[1] // patch_size)
    inputs = keras.Input(shape=img_shape, name="image")
    
    x = build_augmentation()(inputs)
    x = layers.Rescaling(1.0 / 255.0, name="rescaling")(x)
    
    # Patch extraction + Linear projection
    x = layers.Conv2D(projection_dim, kernel_size=patch_size, strides=patch_size, padding="valid", name="patch_conv")(x)
    x = layers.Reshape((num_patches, projection_dim), name="patch_flatten")(x)
    
    # Position embedding
    x = AddPositionEmbs(num_patches=num_patches, projection_dim=projection_dim, name="pos_emb")(x)
    x = layers.Dropout(dropout, name="pos_drop")(x)
    
    # Transformer Encoder blocks
    for i in range(transformer_layers):
        norm1 = layers.LayerNormalization(epsilon=1e-6, name=f"ln1_{i}")(x)
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout, name=f"mha_{i}")(norm1, norm1)
        x = layers.Add(name=f"attn_add_{i}")([x, attn])
        
        norm2 = layers.LayerNormalization(epsilon=1e-6, name=f"ln2_{i}")(x)
        mlp = layers.Dense(mlp_dim, activation="gelu", name=f"mlp1_{i}")(norm2)
        mlp = layers.Dropout(dropout, name=f"mlp_drop_{i}")(mlp)
        mlp = layers.Dense(projection_dim, name=f"mlp2_{i}")(mlp)
        mlp = layers.Dropout(dropout, name=f"mlp_out_drop_{i}")(mlp)
        x = layers.Add(name=f"mlp_add_{i}")([x, mlp])
        
    x = layers.LayerNormalization(epsilon=1e-6, name="head_ln")(x)
    x = layers.GlobalAveragePooling1D(name="seq_pooling")(x)
    x = layers.Dropout(0.2, name="head_dropout")(x)
    
    x = layers.Dense(mlp_dim // 2, activation="gelu", name="head_mlp")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="vit")

# ----------------- 3. MobileNetV2 -----------------
def build_mobilenetv2(num_classes, img_shape=IMG_SHAPE):
    base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=img_shape)
    base.trainable = False
    
    inputs = keras.Input(shape=img_shape, name="image")
    x = build_augmentation()(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_norm")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2"), base

# ----------------- 4. ResNet50 -----------------
@keras.saving.register_keras_serializable(package="plantdisease")
class ResNetPreprocess(layers.Layer):
    _MEAN = (103.939, 116.779, 123.68)
    def call(self, x):
        x = tf.reverse(x, axis=[-1])
        return x - tf.constant(self._MEAN, dtype=x.dtype)
    def compute_output_shape(self, input_shape):
        return input_shape

def build_resnet50(num_classes, img_shape=IMG_SHAPE):
    base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=img_shape)
    base.trainable = False
    
    inputs = keras.Input(shape=img_shape, name="image")
    x = build_augmentation()(inputs)
    x = ResNetPreprocess(name="resnet_norm")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="resnet50"), base

print("All 4 model architectures defined successfully!")


## 🏋️ 6. Training Execution across All 4 Architectures
We train each architecture with callbacks (EarlyStopping, ReduceLROnPlateau), tracking loss/accuracy histories.


In [ ]:
def get_callbacks(patience=5):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(1, patience // 2), min_lr=1e-7, verbose=1),
    ]

def unfreeze_top(base_model, n_layers):
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    for layer in base_model.layers[-n_layers:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

num_classes = len(class_names)
histories = {}
trained_models = {}

# Dictionary to store display names
DISPLAY_NAMES = {
    "cnn": "Custom CNN (from scratch)",
    "vit": "Vision Transformer (ViT from scratch)",
    "mobilenetv2": "MobileNetV2 (transfer learning)",
    "resnet50": "ResNet50 (transfer learning)",
}


### Training Model 1: Custom CNN & Model 2: Vision Transformer (ViT)


In [ ]:
# 1. Train Custom CNN
print("\n" + "="*70 + "\n Training: Custom CNN (from scratch)\n" + "="*70)
cnn_model = build_custom_cnn(num_classes)
cnn_model.compile(optimizer=keras.optimizers.Adam(LR_CNN), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary(line_length=90)

start = time.time()
h_cnn = cnn_model.fit(
    train_ds, validation_data=val_ds, epochs=CNN_EPOCHS,
    class_weight=class_weights, callbacks=get_callbacks(patience=5), verbose=1
)
cnn_time = time.time() - start
cnn_model.save(MODELS_DIR / "cnn.keras")
hist_cnn = {k: [float(v) for v in vals] for k, vals in h_cnn.history.items()}
hist_cnn.update(train_time_sec=cnn_time, epochs_run=len(h_cnn.history["loss"]))
(OUTPUTS_DIR / "cnn_history.json").write_text(json.dumps(hist_cnn, indent=2))
trained_models["cnn"] = cnn_model
histories["cnn"] = hist_cnn

# 2. Train Vision Transformer (ViT)
print("\n" + "="*70 + "\n Training: Vision Transformer (ViT from scratch)\n" + "="*70)
vit_model = build_vit(num_classes)
vit_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=LR_VIT, weight_decay=1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
vit_model.summary(line_length=90)

start = time.time()
h_vit = vit_model.fit(
    train_ds, validation_data=val_ds, epochs=VIT_EPOCHS,
    class_weight=class_weights, callbacks=get_callbacks(patience=5), verbose=1
)
vit_time = time.time() - start
vit_model.save(MODELS_DIR / "vit.keras")
hist_vit = {k: [float(v) for v in vals] for k, vals in h_vit.history.items()}
hist_vit.update(train_time_sec=vit_time, epochs_run=len(h_vit.history["loss"]))
(OUTPUTS_DIR / "vit_history.json").write_text(json.dumps(hist_vit, indent=2))
trained_models["vit"] = vit_model
histories["vit"] = hist_vit


### Training Model 3: MobileNetV2 & Model 4: ResNet50 (Two-Phase Transfer Learning)


In [ ]:
# 3. Train MobileNetV2
print("\n" + "="*70 + "\n Training: MobileNetV2 (Transfer Learning)\n" + "="*70)
mobilenet, base_mobile = build_mobilenetv2(num_classes)
mobilenet.compile(optimizer=keras.optimizers.Adam(LR_HEAD), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

start = time.time()
# Phase 1: Classification Head Only
print("--> Phase 1: Training Classification Head")
h_mb1 = mobilenet.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weights, callbacks=get_callbacks(3), verbose=1)

# Phase 2: Fine-Tuning Backbone
print("--> Phase 2: Fine-Tuning Top Backbone Layers")
unfreeze_top(base_mobile, FINE_TUNE_LAYERS)
mobilenet.compile(optimizer=keras.optimizers.Adam(LR_FINETUNE), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_mb2 = mobilenet.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, class_weight=class_weights, callbacks=get_callbacks(4), verbose=1)

mb_time = time.time() - start
mobilenet.save(MODELS_DIR / "mobilenetv2.keras")
hist_mb = {}
for k in h_mb1.history:
    hist_mb[k] = [float(v) for v in h_mb1.history[k]] + [float(v) for v in h_mb2.history[k]]
hist_mb.update(train_time_sec=mb_time, epochs_run=len(hist_mb["loss"]), phase_boundary=len(h_mb1.history["loss"]))
(OUTPUTS_DIR / "mobilenetv2_history.json").write_text(json.dumps(hist_mb, indent=2))
trained_models["mobilenetv2"] = mobilenet
histories["mobilenetv2"] = hist_mb

# 4. Train ResNet50
print("\n" + "="*70 + "\n Training: ResNet50 (Transfer Learning)\n" + "="*70)
resnet, base_resnet = build_resnet50(num_classes)
resnet.compile(optimizer=keras.optimizers.Adam(LR_HEAD), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

start = time.time()
# Phase 1: Classification Head Only
print("--> Phase 1: Training Classification Head")
h_rn1 = resnet.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weights, callbacks=get_callbacks(3), verbose=1)

# Phase 2: Fine-Tuning Backbone
print("--> Phase 2: Fine-Tuning Top Backbone Layers")
unfreeze_top(base_resnet, FINE_TUNE_LAYERS)
resnet.compile(optimizer=keras.optimizers.Adam(LR_FINETUNE), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_rn2 = resnet.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, class_weight=class_weights, callbacks=get_callbacks(4), verbose=1)

rn_time = time.time() - start
resnet.save(MODELS_DIR / "resnet50.keras")
hist_rn = {}
for k in h_rn1.history:
    hist_rn[k] = [float(v) for v in h_rn1.history[k]] + [float(v) for v in h_rn2.history[k]]
hist_rn.update(train_time_sec=rn_time, epochs_run=len(hist_rn["loss"]), phase_boundary=len(h_rn1.history["loss"]))
(OUTPUTS_DIR / "resnet50_history.json").write_text(json.dumps(hist_rn, indent=2))
trained_models["resnet50"] = resnet
histories["resnet50"] = hist_rn


## 📊 7. Model Evaluation on Held-Out Test Set
We evaluate all 4 trained architectures on the unseen test set, calculating accuracy, precision, recall, macro/weighted F1, top-3 accuracy, model size, and inference latency.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, top_k_accuracy_score, confusion_matrix

results = []
y_true = np.asarray(y_test)

for name, model in trained_models.items():
    print(f"Evaluating {DISPLAY_NAMES[name]}...")
    start = time.time()
    probs = model.predict(test_ds, verbose=0)
    inference_time = (time.time() - start) * 1000 / len(y_true)
    
    y_pred = np.argmax(probs, axis=1)
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    top3_acc = top_k_accuracy_score(y_true, probs, k=3)
    
    model_path = MODELS_DIR / f"{name}.keras"
    size_mb = model_path.stat().st_size / 1e6 if model_path.exists() else 0.0
    params_m = model.count_params() / 1e6
    h = histories.get(name, {})
    
    results.append({
        "Model": DISPLAY_NAMES[name],
        "Test accuracy": round(acc, 4),
        "Precision (macro)": round(p_macro, 4),
        "Recall (macro)": round(r_macro, 4),
        "F1 (macro)": round(f1_macro, 4),
        "F1 (weighted)": round(f1_weighted, 4),
        "Top-3 accuracy": round(top3_acc, 4),
        "Params (M)": round(params_m, 2),
        "Size (MB)": round(size_mb, 1),
        "Inference (ms/img)": round(inference_time, 1),
        "Epochs run": h.get("epochs_run"),
        "Train time (min)": round(h.get("train_time_sec", 0) / 60, 1)
    })

comparison_df = pd.DataFrame(results)
comparison_dir = OUTPUTS_DIR / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(comparison_dir / "metrics_comparison.csv", index=False)

print("\n" + "="*80)
print("FINAL BENCHMARK COMPARISON TABLE")
print("="*80)
display(comparison_df)


## 📈 8. Comparative Visualizations & Learning Curves


In [ ]:
# 1. Validation Curves Overlay
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"cnn": "#d95f02", "vit": "#e7298a", "mobilenetv2": "#1b9e77", "resnet50": "#7570b3"}

for name, h in histories.items():
    epochs = range(1, len(h["val_accuracy"]) + 1)
    axes[0].plot(epochs, h["val_accuracy"], marker="o", label=DISPLAY_NAMES[name], color=colors[name], linewidth=2)
    axes[1].plot(epochs, h["val_loss"], marker="s", label=DISPLAY_NAMES[name], color=colors[name], linewidth=2)

axes[0].set_title("Validation Accuracy across Epochs", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Validation Loss across Epochs", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(comparison_dir / "training_curves_comparison.png", dpi=200)
plt.show()

# 2. Key Metrics Bar Chart
metrics_to_plot = ["Test accuracy", "F1 (macro)", "Top-3 accuracy"]
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
width = 0.2

for i, row in comparison_df.iterrows():
    vals = [row[m] for m in metrics_to_plot]
    ax.bar(x + (i - 1.5) * width, vals, width, label=row["Model"])

ax.set_ylabel("Score", fontsize=11)
ax.set_title("Test Set Metric Comparison by Architecture", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0.7, 1.02)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(comparison_dir / "metrics_comparison.png", dpi=200)
plt.show()


## 🔍 9. Single-Image Clinical Prediction Demo


In [ ]:
# Test prediction on a sample leaf image from the test set
sample_path = X_test[0]
sample_label = y_test[0]

img = Image.open(sample_path)
print(f"Sample Path:  {sample_path}")
print(f"Ground Truth: {class_names[sample_label]}\n")

# Prepare image
raw_arr = tf.image.resize(np.asarray(img.convert('RGB'), dtype=np.float32), IMG_SIZE).numpy()[None, ...]

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"True: {class_names[sample_label]}", fontsize=10)
plt.axis("off")
plt.show()

print(f"{'Model':<35} | {'Top Prediction':<35} | {'Confidence':<10}")
print("-" * 85)
for name, model in trained_models.items():
    preds = model(raw_arr, training=False).numpy()[0]
    top_idx = np.argmax(preds)
    conf = preds[top_idx] * 100
    print(f"{DISPLAY_NAMES[name]:<35} | {class_names[top_idx]:<35} | {conf:.1f}%")


## 📦 10. Package Results for Local Streamlit Web App
This creates `results.zip` containing your trained models and evaluation outputs, ready to download and run with `streamlit run app.py` on your computer.


In [ ]:
results_zip_path = Path("/kaggle/working/results.zip") if os.path.exists("/kaggle") else Path("results.zip")

with zipfile.ZipFile(results_zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("models", "outputs"):
        folder_path = Path(folder)
        if folder_path.exists():
            for p in folder_path.rglob("*"):
                if p.is_file():
                    z.write(p, p.relative_to(folder_path.parent))
    z.writestr("TF_VERSION.txt", f"Trained with TensorFlow {tf.__version__}\n")

zip_size_mb = results_zip_path.stat().st_size / 1e6
print(f"Successfully packaged results to: {results_zip_path} ({zip_size_mb:.2f} MB)")
print("Done! Download results.zip from the right-hand Output panel on Kaggle.")
